# Train MotionSSM V2 — FiLM-conditioned Mamba (Colab T4)

**Architecture changes vs V1:**
- FiLM conditioning: text scale+shift injected at *every* SSM layer (not just input)
- Larger model: d_model=384, d_state=64, n_layers=6 (~55M params, up from 27M)
- max_motion_length=200 (~6.7s clips, up from 3.3s)
- No physics losses during base training
- Dead-channel mask: jaw+eye channels (159:168) excluded from loss

**Runtime:** ~45-60 min/epoch on T4. 200 epochs ≈ 2-3 days.  
Use the **Resume** cell to continue across Colab session restarts.

**Requirements:** 1.6 GB AMASS cache in Google Drive (see Cell 5).


In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/dissertation'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted at {DRIVE_DIR}')


In [ ]:
# ── 2. Clone repo ──────────────────────────────────────────────────────
REPO_URL = 'https://github.com/CatalinButacu/GenAI-Flickr.git'
REPO_DIR = '/content/dissertation'

import os
if not os.path.exists(REPO_DIR):
    !git clone --depth=1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --rebase

%cd {REPO_DIR}
print('Repo:', os.getcwd())


In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')


In [ ]:
# ── 4. Link checkpoint dir to Drive (persists across sessions) ─────────
import os, shutil

DRIVE_DIR = '/content/drive/MyDrive/dissertation'
CKPT_DIR  = f'{DRIVE_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

# Symlink checkpoints/ -> Drive so restarts keep all saved weights
if not os.path.exists('checkpoints'):
    os.symlink(CKPT_DIR, 'checkpoints')
elif not os.path.islink('checkpoints'):
    for f in os.listdir('checkpoints'):
        shutil.move(f'checkpoints/{f}', f'{CKPT_DIR}/{f}')
    shutil.rmtree('checkpoints')
    os.symlink(CKPT_DIR, 'checkpoints')

print(f'checkpoints/ -> {CKPT_DIR}')


In [ ]:
# ── 5. Data setup — AMASS preprocessed cache (~1.6 GB) ─────────────────
#
# Option A (recommended): copy the cache file to your Drive first:
#   MyDrive/dissertation/cache/motion_dataset_4b3c4996b3a3.pkl
#
# Option B: download from AWS S3 (set AWS keys below)
#
import os, shutil

DRIVE_DIR      = '/content/drive/MyDrive/dissertation'
CACHE_FILENAME = 'motion_dataset_4b3c4996b3a3.pkl'
LOCAL_CACHE    = f'data/.cache/{CACHE_FILENAME}'
DRIVE_CACHE    = f'{DRIVE_DIR}/cache/{CACHE_FILENAME}'

os.makedirs('data/.cache', exist_ok=True)
os.makedirs('data/AMASS', exist_ok=True)  # trainer checks this dir exists

if os.path.exists(LOCAL_CACHE):
    print(f'Cache already present ({os.path.getsize(LOCAL_CACHE)/1e9:.2f} GB)')
elif os.path.exists(DRIVE_CACHE):
    print('Copying cache from Drive...')
    shutil.copy2(DRIVE_CACHE, LOCAL_CACHE)
    print(f'Done ({os.path.getsize(LOCAL_CACHE)/1e9:.2f} GB)')
else:
    # Option B: S3 download
    AWS_KEY    = ''  # paste Access Key ID
    AWS_SECRET = ''  # paste Secret Access Key
    S3_URI = 's3://dissertation-motion-cache-91340264/cache/' + CACHE_FILENAME
    if AWS_KEY:
        os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
        os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
        os.environ['AWS_DEFAULT_REGION']    = 'eu-west-1'
        !pip install -q awscli
        !aws s3 cp {S3_URI} {LOCAL_CACHE}
        shutil.copy2(LOCAL_CACHE, DRIVE_CACHE)
        print('Saved copy to Drive for future sessions')
    else:
        raise RuntimeError(
            f'Cache not found.\n'
            f'Upload {CACHE_FILENAME} to Drive at:\n'
            f'  {DRIVE_CACHE}\n'
            f'or set AWS_KEY/AWS_SECRET above.'
        )


## Train MotionSSM V2

FiLM-conditioned bidirectional Mamba — the core text-to-motion model.


In [ ]:
# ── V2 training config ─────────────────────────────────────────────────
# Architecture
D_MODEL  = 384    # V1 was 256
D_STATE  = 64     # V1 was 32
N_LAYERS = 6      # V1 was 4
# Training
EPOCHS   = 200
BATCH    = 64     # T4 16GB fits 64 with gradient-checkpointing
LR       = 3e-4
MAX_LEN  = 200    # 200 frames @ 30fps = 6.7s
CKPT_DIR = 'checkpoints/motion_ssm_v2'
SEED     = 2026

print(f'Architecture: d_model={D_MODEL}, d_state={D_STATE}, n_layers={N_LAYERS}')
print(f'Training:     {EPOCHS} epochs, batch={BATCH}, lr={LR}, max_len={MAX_LEN}')


In [ ]:
# ── Train MotionSSM V2 ─────────────────────────────────────────────────
!python scripts/training/train_motion_ssm.py \
    --data-source          amass \
    --use-sbert \
    --bidirectional \
    --use-film \
    --no-physics-losses \
    --gradient-checkpointing \
    --d-model              {D_MODEL} \
    --d-state              {D_STATE} \
    --n-layers             {N_LAYERS} \
    --max-motion-length    {MAX_LEN} \
    --batch-size           {BATCH} \
    --lr                   {LR} \
    --epochs               {EPOCHS} \
    --num-workers          2 \
    --checkpoint-dir       {CKPT_DIR} \
    --seed                 {SEED} \
    --device               cuda


### Resume after session disconnect

Colab sessions disconnect after ~12 hrs. Re-run cells 1–5 then run the cell below.


In [ ]:
# ── Resume training ────────────────────────────────────────────────────
!python scripts/training/train_motion_ssm.py \
    --data-source          amass \
    --use-sbert \
    --bidirectional \
    --use-film \
    --no-physics-losses \
    --gradient-checkpointing \
    --d-model              {D_MODEL} \
    --d-state              {D_STATE} \
    --n-layers             {N_LAYERS} \
    --max-motion-length    {MAX_LEN} \
    --batch-size           {BATCH} \
    --lr                   {LR} \
    --epochs               {EPOCHS} \
    --num-workers          2 \
    --checkpoint-dir       {CKPT_DIR} \
    --resume               latest \
    --seed                 {SEED} \
    --device               cuda


## Verify Checkpoint


In [ ]:
# ── Inspect best checkpoint ─────────────────────────────────────────────
import torch, os, glob

ck_path = f'{CKPT_DIR}/best_model.pt'
if os.path.isfile(ck_path):
    ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    cfg = ck.get('config')
    print(f'Best: epoch={ck.get("epoch", "?")}, val_loss={ck.get("val_loss", 0.0):.4f}')
    if cfg:
        total = sum(p.numel() for p in ck['model_state_dict'].values())
        print(f'  d_model={cfg.d_model}, d_state={cfg.d_state}, n_layers={cfg.n_layers}')
        print(f'  params={total:,}  use_film={getattr(cfg, "use_film", False)}')
else:
    ckpts = sorted(glob.glob(f'{CKPT_DIR}/epoch_*.pt'))
    print(f'No best_model.pt yet. Epoch checkpoints: {len(ckpts)}')
    if ckpts:
        print('Latest:', ckpts[-1])


## Quick Inference Test


In [ ]:
# ── Test generation on sample prompts ──────────────────────────────────
import sys, torch
sys.path.insert(0, '.')

from src.modules.motion.config import GeneratorConfig, MotionStrategy
from src.modules.motion.generator import MotionGenerator

gen = MotionGenerator(GeneratorConfig(
    strategy=MotionStrategy.SSM,
    checkpoint_path=f'{CKPT_DIR}/best_model.pt',
))

for prompt in [
    'a person walks forward',
    'a person runs and jumps',
    'a person sits down slowly',
]:
    motion = gen.generate(prompt, duration=2.0, fps=30)
    print(f'{prompt!r:45s} -> {motion.shape}')
